# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0202

In [15]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

In [16]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Tabular_Models_Dataset\community_index_v9.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target stats — mean: {df['community_suppression'].mean():.3f}  "
      f"std: {df['community_suppression'].std():.3f}  "
      f"range: [{df['community_suppression'].min():.3f}, {df['community_suppression'].max():.3f}]")

Dataset shape: 48,000 rows × 168 columns
Target stats — mean: 0.670  std: 0.377  range: [0.000, 0.999]


In [17]:
r0_cols = [f"R0_c{k}" for k in range(1, 21)]

active_cols = [f"active_c{k}" for k in range(1, 21)]

structural_cols = ['community_size', 'diversity_level']

feature_cols = r0_cols + active_cols + structural_cols

print(f"Feature count:          {len(feature_cols)}  (vs 142 in the composition model)")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  Structural:           {len(structural_cols)}")
print(f"  Species membership:   0  ← intentionally excluded")


X = df[feature_cols]
y = df['community_suppression']

print(f"\nX shape: {X.shape}    y shape: {y.shape}")

Feature count:          42  (vs 142 in the composition model)
  R0 concentrations:    20
  Active flags:         20
  Structural:           2
  Species membership:   0  ← intentionally excluded

X shape: (48000, 42)    y shape: (48000,)


In [18]:
# train test split

strat_key = (df['community_size'].astype(str) + "_" +
             df['diversity_level'].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)

print(f"Train: {X_train.shape[0]:,} rows   Test: {X_test.shape[0]:,} rows")

Train: 38,400 rows   Test: 9,600 rows


In [19]:
# baseline models

y_mean = y_train.mean()

y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
print(f"Global mean predictor: R² = {r2_score(y_test, y_pred_mean):.4f}  (always 0.0 by definition)")

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)
r2_ridge  = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
print(f"Ridge Regression:      R² = {r2_ridge:.4f}   MAE = {mae_ridge:.4f}")

Global mean predictor: R² = -0.0002  (always 0.0 by definition)
Ridge Regression:      R² = 0.8392   MAE = 0.1015


In [20]:
# xgboost initial


model_init = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=7,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    min_child_weight=3,

    reg_lambda=1.0,

    early_stopping_rounds=30,

    random_state=42,
    n_jobs=-1,       
    eval_metric='rmse',
    verbosity=0
)


model_init.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_init = model_init.predict(X_test)
r2_init  = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)

print(f"XGBoost initial:  R² = {r2_init:.4f}   MAE = {mae_init:.4f}")
print(f"Best iteration: {model_init.best_iteration}")

XGBoost initial:  R² = 0.8624   MAE = 0.0787
Best iteration: 208


In [7]:
# hyperparameter tuning

param_grid = {
    'max_depth':        [5, 6, 7, 8, 9],

    'learning_rate':    [0.1,0.05,0.1],
    'subsample':        [0.6,0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6,0.8,0.9,1.0],
}
# Total combinations: 3^4 = 81. With cv=3 folds: 81 × 3 = 243 model fits.

# KFold(n_splits=3, shuffle=True): randomise row order before splitting.
# Critical here: without shuffle, folds might be ordered by broth_id and the
# model could see all D=1 broths in one fold — giving misleading CV scores.
cv = KFold(n_splits=3, shuffle=True, random_state=42)

base_est = xgb.XGBRegressor(
    n_estimators=200,   # kept low for speed during search
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    verbosity=0
)

# GridSearchCV:
#   scoring='neg_mean_squared_error': sklearn always maximises the score,
#   so we negate MSE (lower MSE = higher neg_MSE = better).
#   refit=True: after finding best params, automatically refit on full X_train.
grid_search = GridSearchCV(
    estimator=base_est,
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    refit=True
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k}: {v}")

# np.sqrt(-grid_search.best_score_): we stored negative MSE, so negate
# to get MSE, then take square root to get RMSE in suppression units.
best_cv_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best CV RMSE: {best_cv_rmse:.4f}")


SECTION 7: Hyperparameter search (GridSearchCV)
Searching... (this takes a few minutes)
Fitting 3 folds for each of 300 candidates, totalling 900 fits

Best parameters:
  colsample_bytree: 0.8
  learning_rate: 0.05
  max_depth: 6
  subsample: 0.9
Best CV RMSE: 0.1404


In [21]:
# final model


model_final = xgb.XGBRegressor(
    **grid_search.best_params_,
    n_estimators=1000,         # high ceiling; early stopping decides when to stop
    min_child_weight=3,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbosity=0
)

model_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

y_pred_final = model_final.predict(X_test)

r2_final   = r2_score(y_test, y_pred_final)
mae_final  = mean_absolute_error(y_test, y_pred_final)
mse_final  = mean_squared_error(y_test, y_pred_final)
rmse_final = np.sqrt(mse_final)

print(f"\n--- Nutrient-Only Model Performance ---")
print(f"R²:   {r2_final:.4f}")
print(f"MAE:  {mae_final:.4f}")
print(f"RMSE: {rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration}")

print(f"\n--- Model Comparison ---")
print(f"{'Model':<35} {'R²':>8}  {'MAE':>8}")
print("-" * 55)
print(f"{'Global mean baseline':<35} {'0.0000':>8}  {'N/A':>8}")
print(f"{'Ridge (nutrients only)':<35} {r2_ridge:>8.4f}  {mae_ridge:>8.4f}")
print(f"{'XGBoost (nutrients only)':<35} {r2_final:>8.4f}  {mae_final:>8.4f}")


[0]	validation_0-rmse:0.36368
[50]	validation_0-rmse:0.15236
[100]	validation_0-rmse:0.14122
[150]	validation_0-rmse:0.14049
[200]	validation_0-rmse:0.14042
[250]	validation_0-rmse:0.14042
[267]	validation_0-rmse:0.14043

--- Nutrient-Only Model Performance ---
R²:   0.8624
MAE:  0.0789
RMSE: 0.1404
Trees used: 217

--- Model Comparison ---
Model                                     R²       MAE
-------------------------------------------------------
Global mean baseline                  0.0000       N/A
Ridge (nutrients only)                0.8392    0.1015
XGBoost (nutrients only)              0.8624    0.0789


In [22]:
# per condition breakdown

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

print("\nBy diversity level:")
print(f"{'D':<5} {'R²':>8}  {'MAE':>8}  {'N':>6}")
print("-" * 30)
# .groupby('column'): splits the DataFrame into groups, one per unique value.
# Each iteration gives (group_name, sub_dataframe).
for d, grp in results_df.groupby('diversity_level'):
    r2_d  = r2_score(grp['y_true'], grp['y_pred'])
    mae_d = mean_absolute_error(grp['y_true'], grp['y_pred'])
    print(f"D={d:<4} {r2_d:>8.4f}  {mae_d:>8.4f}  {len(grp):>6,}")

print("\nBy community size:")
print(f"{'N':<5} {'R²':>8}  {'MAE':>8}  {'Count':>6}")
print("-" * 30)
for n, grp in results_df.groupby('community_size'):
    r2_n  = r2_score(grp['y_true'], grp['y_pred'])
    mae_n = mean_absolute_error(grp['y_true'], grp['y_pred'])
    print(f"N={n:<4} {r2_n:>8.4f}  {mae_n:>8.4f}  {len(grp):>6,}")


By diversity level:
D           R²       MAE       N
------------------------------
D=1      0.8941    0.0719   3,000
D=2      0.8784    0.0758   2,400
D=4      0.3573    0.0799   1,800
D=8      0.6220    0.0828   1,200
D=16     0.4364    0.0970   1,200

By community size:
N           R²       MAE   Count
------------------------------
N=12     0.6923    0.1395   1,920
N=16     0.7799    0.1116   1,920
N=24     0.9087    0.0653   1,920
N=32     0.9259    0.0509   1,920
N=48     0.9787    0.0270   1,920


In [23]:
# shap analysis

print('shap analysis')


explainer = shap.TreeExplainer(model_final)

rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=2000, replace=False)


X_sample = X_test.iloc[idx]
shap_values = explainer.shap_values(X_sample)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nAll features ranked by mean |SHAP|:")
print(shap_importance.round(5).to_string())

# Group totals
r0_shap     = shap_importance[r0_cols].sum()
active_shap = shap_importance[active_cols].sum()
struct_shap = shap_importance[structural_cols].sum()
total_shap  = r0_shap + active_shap + struct_shap

print(f"\nSHAP by group:")
print(f"  R0 concentrations:  {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:       {active_shap:.4f}  ({100*active_shap/total_shap:.1f}%)")
print(f"  Structural:         {struct_shap:.4f}  ({100*struct_shap/total_shap:.1f}%)")

shap analysis

All features ranked by mean |SHAP|:
R0_c4              0.11052
community_size     0.06144
R0_c11             0.05095
R0_c1              0.04958
R0_c17             0.04413
R0_c19             0.01453
R0_c8              0.01366
diversity_level    0.01265
R0_c2              0.01193
R0_c16             0.01186
R0_c3              0.01139
R0_c15             0.01035
R0_c7              0.00764
active_c1          0.00704
R0_c12             0.00633
R0_c9              0.00579
R0_c13             0.00481
R0_c14             0.00417
active_c17         0.00316
R0_c10             0.00294
active_c11         0.00266
active_c7          0.00263
active_c4          0.00213
active_c19         0.00131
R0_c20             0.00128
active_c14         0.00084
active_c12         0.00083
active_c8          0.00081
active_c15         0.00078
R0_c5              0.00069
R0_c6              0.00045
active_c13         0.00037
active_c16         0.00020
active_c10         0.00005
R0_c18             0.00004
acti

In [24]:
# =============================================================================
# SECTION 11 — VISUALISATION (6 PANELS)
# =============================================================================

print("\n" + "=" * 60)
print("SECTION 11: Generating plots → nutrient_only_results.png")
print("=" * 60)

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'XGBoost NUTRIENT-ONLY Suppression Predictor — MiCRM v8\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
    f'[Compare: composition+nutrient model R²=0.930]',
    fontsize=12, fontweight='bold', y=0.99
)

# Panel 1: Predicted vs Actual
ax1 = fig.add_subplot(2, 3, 1)
ax1.scatter(y_test, y_pred_final, alpha=0.1, s=2, c='steelblue')
ax1.plot([0, 1], [0, 1], 'r--', lw=1.5, label='Perfect prediction')
# Colour points by diversity level to show which conditions are hard to predict
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = df.loc[X_test.index, 'diversity_level'].values == d
    ax1.scatter(y_test.values[mask], y_pred_final[mask],
                alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted suppression')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

# Panel 2: Residuals coloured by community size
ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - y_pred_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = df.loc[X_test.index, 'community_size'].values == n
    ax2.scatter(y_pred_final[mask], resid[mask],
                alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

# Panel 3: SHAP importance — all 42 features
ax3 = fig.add_subplot(2, 3, 3)
# Colour: green for R0, orange for active flags, purple for structural
def feat_colour(f):
    if f in r0_cols:       return '#4CAF50'
    if f in active_cols:   return '#FF9800'
    return '#9C27B0'

top_n = min(25, len(feature_cols))
top_shap = shap_importance.head(top_n)
cols = [feat_colour(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title(f'Top {top_n} Features by SHAP')
from matplotlib.patches import Patch
ax3.legend(handles=[
    Patch(fc='#4CAF50', label='R0 concentration'),
    Patch(fc='#FF9800', label='Active flag'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=8)

# Panel 4: R² by diversity level
ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# Panel 5: R² by community size
ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N')
ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# Panel 6: SHAP group pie chart
ax6 = fig.add_subplot(2, 3, 6)
ax6.pie(
    [r0_shap, active_shap, struct_shap],
    labels=['R0\nconcentrations', 'Active\nflags', 'Structural\n(size/div)'],
    colors=['#4CAF50', '#FF9800', '#9C27B0'],
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax6.set_title('SHAP Importance by Feature Group\n(nutrient-only model)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('nutrient_only_results_updated02.png', dpi=150, bbox_inches='tight')
# bbox_inches='tight': trims whitespace from the saved image.
plt.close()
print("Saved: nutrient_only_results_updated02.png")


SECTION 11: Generating plots → nutrient_only_results.png
Saved: nutrient_only_results_updated02.png


In [ ]:
# =============================================================================
# SECTION 12 — SAVE OUTPUTS
# =============================================================================

print("\n" + "=" * 60)
print("SECTION 12: Saving outputs")
print("=" * 60)

# Save model in XGBoost's native JSON format.
# Reload later with: m = xgb.XGBRegressor(); m.load_model('...')
model_final.save_model('xgb_nutrient_only_v8.json')
print("Model saved: xgb_nutrient_only_v8.json")

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'residual':        y_test.values - y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
}).to_csv('predictions_nutrient_only_v8.csv', index=False)
print("Predictions saved: predictions_nutrient_only_v8.csv")